# 03 — Run dbt (Gold Layer)
**Pipeline:** Silver Delta → dbt models → Gold Delta tables (Unity Catalog)  
**Layer:** Gold (aggregated mart tables — mart_daily_volume, mart_risk_exposure, mart_top_instruments)  
**Runs on:** Databricks cluster (dbt connects via HTTP path)  
**Called by:** `flows/trade_pipeline_flow.py` → `run_dbt` + `test_dbt` tasks

## Cell 1 — Install dbt-databricks on the cluster

In [ ]:
# Install dbt-databricks — required every cluster start on Community Edition
%pip install dbt-databricks --quiet

import sys
import os
import subprocess

REPO_ROOT = "/Workspace/Users/pravash.cse@gmail.com/trade-analytics-lakehouse"
DBT_PROJECT_DIR = f"{REPO_ROOT}/dbt_project"

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# Verify dbt is installed
result = subprocess.run(["dbt", "--version"], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

## Cell 2 — Set dbt environment variables

In [ ]:
# These are read by dbt_project/profiles.yml via env_var()
# In production these are set in cluster environment variables
# Here we set them explicitly for notebook runs

# Get token from Databricks secret or set directly
# Replace with your actual values or use dbutils.secrets
os.environ["DATABRICKS_HOST"]      = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
os.environ["DATABRICKS_TOKEN"]     = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
os.environ["DATABRICKS_HTTP_PATH"] = "/sql/1.0/warehouses/fc7181f319f9ab14"

print(f"DATABRICKS_HOST      : {os.environ['DATABRICKS_HOST']}")
print(f"DATABRICKS_TOKEN     : {'*' * 20} (set)")
print(f"DATABRICKS_HTTP_PATH : {os.environ['DATABRICKS_HTTP_PATH']}")

# Verify dbt project directory exists
if os.path.exists(DBT_PROJECT_DIR):
    print(f"\ndbt project found at: {DBT_PROJECT_DIR} ✓")
    print(f"Contents: {os.listdir(DBT_PROJECT_DIR)}")
else:
    print(f"\ndbt project NOT found at: {DBT_PROJECT_DIR}")
    print("Check that REPO_ROOT is correct and the dbt_project/ folder exists")

## Cell 3 — Verify Silver table is readable (dbt input)

In [ ]:
from src.trade_analytics.config.settings import SILVER_PATH

df_silver = spark.read.format("delta").load(SILVER_PATH)
print(f"[Silver] Row count       : {df_silver.count():,}")
print(f"[Silver] Column count    : {len(df_silver.columns)}")
print(f"[Silver] Columns         : {df_silver.columns}")

# Verify the columns dbt staging model expects
required_cols = [
    "trade_id", "trader_id", "instrument", "direction",
    "notional_eur", "desk", "region", "trade_date",
    "risk_tier", "velocity_flag", "alert_level", "is_anomaly"
]
missing = [c for c in required_cols if c not in df_silver.columns]
if missing:
    print(f"\n[WARNING] Missing columns expected by dbt: {missing}")
    print("Run notebook 02 first to regenerate Silver")
else:
    print("\nAll required Silver columns present ✓")

## Cell 4 — dbt debug (verify connection to Databricks SQL)

In [ ]:
def run_dbt_command(cmd: list, label: str) -> bool:
    """Runs a dbt command, prints output, returns True if successful."""
    print(f"\n{'='*50}")
    print(f"  {label}")
    print(f"  Command: dbt {' '.join(cmd[1:])}")
    print(f"{'='*50}")

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        cwd=DBT_PROJECT_DIR
    )

    print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr)

    success = result.returncode == 0
    status  = "✓ PASSED" if success else "✗ FAILED"
    print(f"\nResult: {status}")
    return success


# Verify dbt can connect to Databricks
debug_ok = run_dbt_command(
    ["dbt", "debug", "--profiles-dir", "."],
    "dbt debug — verify connection"
)

if not debug_ok:
    print("\nFix the connection issue before proceeding.")
    print("Common causes:")
    print("  1. DATABRICKS_HTTP_PATH is wrong — get it from SQL Warehouses → your warehouse → Connection details")
    print("  2. Cluster is not running")
    print("  3. profiles.yml env_var names don't match os.environ keys set in Cell 2")

## Cell 5 — dbt compile (validates all SQL syntax)

In [ ]:
compile_ok = run_dbt_command(
    ["dbt", "compile", "--profiles-dir", "."],
    "dbt compile — validate SQL syntax"
)

if compile_ok:
    # Show compiled SQL for the staging model as a sanity check
    compiled_path = f"{DBT_PROJECT_DIR}/target/compiled/trade_analytics/models/staging/stg_trades.sql"
    if os.path.exists(compiled_path):
        with open(compiled_path) as f:
            print("\n── Compiled stg_trades.sql ──────────────────────────")
            print(f.read())

## Cell 6 — dbt run (materialise all models)

In [ ]:
run_ok = run_dbt_command(
    ["dbt", "run", "--profiles-dir", "."],
    "dbt run — materialise staging + intermediate + mart models"
)

if not run_ok:
    raise Exception("dbt run failed — check output above before proceeding to tests")

## Cell 7 — dbt test (run all schema + custom tests)

In [ ]:
test_ok = run_dbt_command(
    ["dbt", "test", "--profiles-dir", "."],
    "dbt test — schema.yml + assert_positive_notional.sql"
)

if not test_ok:
    raise Exception(
        "dbt tests FAILED — Gold layer has data quality issues. "
        "export_gold_csv will NOT run until tests pass."
    )

print("\nAll dbt tests passed ✓ — Gold layer is clean")

## Cell 8 — Verify Gold mart tables in Unity Catalog

In [ ]:
from src.trade_analytics.config.settings import GOLD_PATH

gold_tables = [
    "mart_daily_volume",
    "mart_risk_exposure",
    "mart_top_instruments",
]

print("── Gold Mart Tables ──────────────────────────────────")
for table in gold_tables:
    try:
        df = spark.sql(f"SELECT * FROM trade_analytics_gold.{table}")
        print(f"  {table:<30} : {df.count():>6,} rows ✓")
    except Exception as e:
        print(f"  {table:<30} : NOT FOUND — {e}")

print("")
display(spark.sql("SHOW TABLES IN trade_analytics_gold"))

## Cell 9 — Preview each Gold mart

In [ ]:
print("── mart_daily_volume (top 5 rows) ───────────────────")
display(spark.sql("""
    SELECT *
    FROM trade_analytics_gold.mart_daily_volume
    ORDER BY trade_date DESC, total_notional_eur DESC
    LIMIT 5
"""))

print("\n── mart_risk_exposure (top 5 traders by net exposure) ─")
display(spark.sql("""
    SELECT trader_id, desk, net_exposure_eur, exposure_risk_tier, worst_alert_level
    FROM trade_analytics_gold.mart_risk_exposure
    ORDER BY ABS(net_exposure_eur) DESC
    LIMIT 5
"""))

print("\n── mart_top_instruments (ranked by volume) ──────────")
display(spark.sql("""
    SELECT instrument, desk, trade_count, total_volume_eur, anomaly_rate_pct, volume_rank
    FROM trade_analytics_gold.mart_top_instruments
    ORDER BY volume_rank
"""))

## Cell 10 — Export Gold tables to CSV (for Streamlit dashboard)

In [ ]:
import pandas as pd

VOLUME_ROOT   = "/Volumes/workspace/default/trade-analytics"
CSV_OUTPUT    = f"{VOLUME_ROOT}/dashboard_data"

# Create dashboard_data folder in Volume
dbutils.fs.mkdirs(CSV_OUTPUT)

export_tables = [
    "mart_daily_volume",
    "mart_risk_exposure",
    "mart_top_instruments",
]

for table in export_tables:
    df = spark.sql(f"SELECT * FROM trade_analytics_gold.{table}")

    # Write as single CSV partition to Volume
    csv_path = f"{CSV_OUTPUT}/{table}.csv"
    (
        df.coalesce(1)
          .write
          .mode("overwrite")
          .option("header", "true")
          .csv(f"{CSV_OUTPUT}/{table}_tmp")
    )

    # Find the part file and rename it
    part_files = [f.path for f in dbutils.fs.ls(f"{CSV_OUTPUT}/{table}_tmp") if f.name.startswith("part-")]
    if part_files:
        dbutils.fs.cp(part_files[0], csv_path)
        dbutils.fs.rm(f"{CSV_OUTPUT}/{table}_tmp", recurse=True)
        print(f"  Exported → {csv_path} ✓")
    else:
        print(f"  No part file found for {table}")

print("\nCSV export complete ✓")
display(dbutils.fs.ls(CSV_OUTPUT))

## Cell 11 — Generate dbt docs

In [ ]:
# Generate dbt documentation (lineage graph)
# Screenshot the output for your README and portfolio
run_dbt_command(
    ["dbt", "docs", "generate", "--profiles-dir", "."],
    "dbt docs generate — build lineage graph"
)

# Note: dbt docs serve won't work inside a Databricks notebook
# To view the docs locally:
# 1. Download target/catalog.json and target/manifest.json
# 2. Run locally: dbt docs serve
# 3. Open http://localhost:8080

docs_path = f"{DBT_PROJECT_DIR}/target/catalog.json"
if os.path.exists(docs_path):
    print(f"\ndbt docs generated ✓")
    print(f"catalog.json : {docs_path}")
    print(f"manifest.json: {DBT_PROJECT_DIR}/target/manifest.json")
    print("\nDownload both files and run 'dbt docs serve' locally to view the lineage graph")
else:
    print("docs generation may have failed — check output above")

## Cell 12 — Full pipeline summary

In [ ]:
from src.trade_analytics.config.settings import BRONZE_PATH, SILVER_PATH, GOLD_PATH

bronze_count = spark.read.format("delta").load(BRONZE_PATH).count()
silver_count = spark.read.format("delta").load(SILVER_PATH).count()

print("\n══════════════════════════════════════════════════════")
print("  Trade Analytics Lakehouse — Pipeline Summary        ")
print("══════════════════════════════════════════════════════")
print(f"  Bronze rows (raw validated)      : {bronze_count:>8,}")
print(f"  Silver rows (enriched + filtered): {silver_count:>8,}")
print(f"  Filtered out (CANCELLED)         : {bronze_count - silver_count:>8,}")
print()
print("  Gold mart tables:")
for table in gold_tables:
    try:
        count = spark.sql(f"SELECT COUNT(*) FROM trade_analytics_gold.{table}").collect()[0][0]
        print(f"    {table:<35}: {count:>6,} rows")
    except:
        print(f"    {table:<35}: not found")
print()
print("  Medallion layers:")
print(f"    Bronze  → {BRONZE_PATH}")
print(f"    Silver  → {SILVER_PATH}")
print(f"    Gold    → trade_analytics_gold.*  (Unity Catalog)")
print()
print("  dbt tests    : ALL PASSED ✓")
print("  CSV exports  : dashboard/data/*.csv ✓")
print("══════════════════════════════════════════════════════")